# Part 6. 하이퍼파라미터 실험 (Grid Search)

탐색할 파라미터:
- **학습률 (lr)**: 1e-3 / 1e-4 / 1e-5
- **에포크 수**: 10 / 20
- **옵티마이저**: Adam / SGD(momentum=0.9)

총 12가지 조합을 순차 실행하여 Test Accuracy 기준 최적 파라미터를 찾는다.

> **이 파일 단독으로 실행 가능** — 셀을 위에서 아래로 순서대로 실행할 것.

## Setup — 라이브러리 & 시드 & 경로

In [ ]:
import os, random, glob, time
from itertools import product

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from facenet_pytorch import MTCNN

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 시드 고정 ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── 경로 ──────────────────────────────────────────────
ROOT_PATH    = r'C:\Users\user\Documents\50.2026\53.DL_실습\mid_test_image\raw_data'
IMAGE_FOLDER = os.path.join(ROOT_PATH, 'Video', 'Images')
LABEL_PATH   = os.path.join(ROOT_PATH, 'mosi_audio_metadata.csv')

print(f'device: {device}')
print(f'torch : {torch.__version__}')
print(f'CUDA  : {torch.cuda.is_available()}')

## Setup — 전처리 함수

In [ ]:
mtcnn = MTCNN(image_size=224, margin=20, keep_all=False, device=device)

def face_crop_mtcnn(img):
    boxes, probs = mtcnn.detect(img)
    if boxes is not None:
        box = boxes[0].astype(int)
        return img.crop(box), True
    return img, False

def enhance_face(pil_img):
    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    gaussian = cv2.GaussianBlur(img, (0, 0), 2.0)
    img = cv2.addWeighted(img, 1.5, gaussian, -0.5, 0)
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

print('전처리 함수 준비 완료')

## Setup — 데이터셋 & DataLoader

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class FaceDataset(Dataset):
    def __init__(self, fnames, labels, img_dir, transform=None):
        self.fnames    = fnames
        self.labels    = labels
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        label = self.labels[idx]
        img_path = fname if os.path.isabs(fname) else os.path.join(self.img_dir, fname)
        img = Image.open(img_path).convert('RGB')
        img, _ = face_crop_mtcnn(img)
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        img = img.resize((224, 224))
        img = enhance_face(img)
        if self.transform:
            img = self.transform(img)
        return img, label

# 라벨 로드
label_df   = pd.read_csv(LABEL_PATH)
label_dict = {os.path.splitext(r['file_name'])[0]: r['sentiment'] for _, r in label_df.iterrows()}

all_files = (
    glob.glob(os.path.join(IMAGE_FOLDER, '**', '*.jpg'),  recursive=True) +
    glob.glob(os.path.join(IMAGE_FOLDER, '**', '*.jpeg'), recursive=True) +
    glob.glob(os.path.join(IMAGE_FOLDER, '**', '*.png'),  recursive=True)
)

fnames_all, labels_all = [], []
for fpath in all_files:
    key = os.path.splitext(os.path.basename(fpath))[0]
    if key in label_dict:
        fnames_all.append(fpath)
        labels_all.append(1 if label_dict[key] >= 0 else 0)

fnames_train, fnames_test, labels_train, labels_test = train_test_split(
    fnames_all, labels_all, test_size=0.2, random_state=SEED, stratify=labels_all
)

class_counts = [labels_train.count(0), labels_train.count(1)]
weights      = [1.0 / class_counts[l] for l in labels_train]
sampler      = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_dataset = FaceDataset(fnames_train, labels_train, IMAGE_FOLDER, transform=train_transform)
test_dataset  = FaceDataset(fnames_test,  labels_test,  IMAGE_FOLDER, transform=test_transform)
train_loader  = DataLoader(train_dataset, batch_size=32, sampler=sampler)
test_loader   = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f'train: {len(fnames_train)}, test: {len(fnames_test)}')
print('DataLoader 준비 완료')

## Setup — 학습 / 평가 함수

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs, device):
    train_loss_list, train_acc_list = [], []
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, predicted  = torch.max(outputs, 1)
            correct      += (predicted == labels).sum().item()
            total        += labels.size(0)
        epoch_loss = running_loss / total
        epoch_acc  = correct / total
        train_loss_list.append(epoch_loss)
        train_acc_list.append(epoch_acc)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch [{epoch+1}/{num_epochs}]  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')
    return train_loss_list, train_acc_list


def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted  = torch.max(outputs, 1)
            correct      += (predicted == labels).sum().item()
            total        += labels.size(0)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels


print('train_model / evaluate_model 정의 완료')

---
## 6-1. 실험 설정 정의

> 빠르게 테스트하려면 `EPOCH_LIST = [5, 10]` 으로 줄일 것.

In [ ]:
LR_LIST        = [1e-3, 1e-4, 1e-5]
EPOCH_LIST     = [10, 20]
OPTIMIZER_LIST = ['Adam', 'SGD']

grid_configs = [
    {'lr': lr, 'num_epochs': ep, 'optimizer': opt}
    for lr, ep, opt in product(LR_LIST, EPOCH_LIST, OPTIMIZER_LIST)
]

print(f'총 실험 수: {len(grid_configs)}')
for i, cfg in enumerate(grid_configs):
    print(f'  [{i+1:02d}] {cfg}')

## 6-2. 실험 루프 실행

In [ ]:
# ── 함수 재정의 (커널에 남은 이전 버전 덮어쓰기) ──────────────
def train_model(model, train_loader, criterion, optimizer, num_epochs, device):
    train_loss_list, train_acc_list = [], []
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, predicted  = torch.max(outputs, 1)
            correct      += (predicted == labels).sum().item()
            total        += labels.size(0)
        epoch_loss = running_loss / total
        epoch_acc  = correct / total
        train_loss_list.append(epoch_loss)
        train_acc_list.append(epoch_acc)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch [{epoch+1}/{num_epochs}]  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')
    return train_loss_list, train_acc_list

def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted  = torch.max(outputs, 1)
            correct      += (predicted == labels).sum().item()
            total        += labels.size(0)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

def build_model():
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    return m.to(device)

def build_optimizer(model, opt_name, lr):
    if opt_name == 'Adam':
        return optim.Adam(model.parameters(), lr=lr)
    return optim.SGD(model.parameters(), lr=lr, momentum=0.9)

# ── 실험 루프 ─────────────────────────────────────────────────
grid_results = []
criterion    = nn.CrossEntropyLoss()

for i, cfg in enumerate(grid_configs):
    lr, num_ep, opt_name = cfg['lr'], cfg['num_epochs'], cfg['optimizer']

    print(f'\n[{i+1}/{len(grid_configs)}] lr={lr}  epochs={num_ep}  optimizer={opt_name}')
    print('-' * 50)

    exp_model = build_model()
    exp_optim = build_optimizer(exp_model, opt_name, lr)
    t0 = time.time()

    exp_train_loss, exp_train_acc = train_model(
        exp_model, train_loader, criterion, exp_optim, num_ep, device
    )
    test_loss_e, test_acc_e, preds_e, labels_e = evaluate_model(
        exp_model, test_loader, criterion, device
    )

    elapsed = time.time() - t0
    f1 = f1_score(labels_e, preds_e, average='weighted')
    print(f'  → Test Acc: {test_acc_e:.4f}  F1: {f1:.4f}  ({elapsed:.0f}s)')

    grid_results.append({
        'optimizer'      : opt_name,
        'lr'             : lr,
        'num_epochs'     : num_ep,
        'test_acc'       : round(test_acc_e, 4),
        'f1_score'       : round(f1, 4),
        'final_train_acc': round(exp_train_acc[-1], 4),
        'elapsed_sec'    : round(elapsed, 1),
    })

    del exp_model
    torch.cuda.empty_cache()

print('\n=== 전체 실험 완료 ===')

## 6-3. 결과 테이블 & 최적 파라미터

In [ ]:
results_df = pd.DataFrame(grid_results).sort_values('test_acc', ascending=False).reset_index(drop=True)
results_df.index += 1

print('=== 실험 결과 (Test Accuracy 내림차순) ===')
print(results_df.to_string())

best = results_df.iloc[0]
print(f'\n★ 최적 파라미터 ★')
print(f'  optimizer : {best.optimizer}')
print(f'  lr        : {best.lr}')
print(f'  epochs    : {int(best.num_epochs)}')
print(f'  test_acc  : {best.test_acc:.4f}')
print(f'  f1_score  : {best.f1_score:.4f}')

## 6-4. 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('하이퍼파라미터 실험 결과', fontsize=14)

# ── 그래프 1: 전체 조합 Bar Chart ───────────────────────
df_plot = results_df.copy()
df_plot['label'] = df_plot.apply(
    lambda r: f"{r['optimizer']}\nlr={r['lr']:.0e}\nep={int(r['num_epochs'])}", axis=1
)
colors_bar = ['tomato' if i == 0 else 'steelblue' for i in range(len(df_plot))]
axes[0].bar(df_plot.index, df_plot['test_acc'], color=colors_bar)
axes[0].set_xticks(df_plot.index)
axes[0].set_xticklabels(df_plot['label'], fontsize=7)
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('전체 조합 비교 (빨간색=최고)')
for idx, row in df_plot.iterrows():
    axes[0].text(idx, row['test_acc'] + 0.005, f"{row['test_acc']:.3f}",
                 ha='center', va='bottom', fontsize=7)

# ── 그래프 2: lr × optimizer 히트맵 (가장 큰 epoch 기준) ─
pivot_df = results_df[results_df['num_epochs'] == EPOCH_LIST[-1]].pivot_table(
    index='lr', columns='optimizer', values='test_acc'
)
im = axes[1].imshow(pivot_df.values, cmap='YlOrRd', aspect='auto', vmin=0.5, vmax=1.0)
axes[1].set_xticks(range(len(pivot_df.columns)))
axes[1].set_yticks(range(len(pivot_df.index)))
axes[1].set_xticklabels(pivot_df.columns)
axes[1].set_yticklabels([f'{v:.0e}' for v in pivot_df.index])
axes[1].set_xlabel('Optimizer')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title(f'lr × optimizer 히트맵\n(epochs={EPOCH_LIST[-1]})')
plt.colorbar(im, ax=axes[1])
for r in range(pivot_df.shape[0]):
    for c in range(pivot_df.shape[1]):
        val = pivot_df.values[r, c]
        axes[1].text(c, r, f'{val:.3f}', ha='center', va='center', fontsize=10,
                     color='black' if val < 0.75 else 'white')

# ── 그래프 3: epoch별 정확도 변화 (최적 lr 기준) ──────────
best_lr = best['lr']
df_adam = results_df[(results_df['optimizer'] == 'Adam') & (results_df['lr'] == best_lr)].sort_values('num_epochs')
df_sgd  = results_df[(results_df['optimizer'] == 'SGD')  & (results_df['lr'] == best_lr)].sort_values('num_epochs')
axes[2].plot(df_adam['num_epochs'], df_adam['test_acc'], marker='o', color='tomato',    linewidth=2, label=f'Adam lr={best_lr:.0e}')
axes[2].plot(df_sgd['num_epochs'],  df_sgd['test_acc'],  marker='s', color='steelblue', linewidth=2, linestyle='--', label=f'SGD lr={best_lr:.0e}')
axes[2].axhline(y=0.641, color='gray', linestyle=':', linewidth=1.5, label='AlexNet baseline 0.641')
axes[2].set_xlabel('Epochs')
axes[2].set_ylabel('Test Accuracy')
axes[2].set_title(f'Epoch 수에 따른 정확도 변화\n(lr={best_lr:.0e})')
axes[2].legend(fontsize=8)
axes[2].grid(True)

plt.tight_layout()
plt.savefig('viz_05_grid_search.png', dpi=150)
plt.show()
print('viz_05_grid_search.png 저장 완료')